# Diabetic Retinopathy Detection Using Convolutional Neural Networks
### B.Tech CSE Project-Based Learning (PBL) | GM University

---

> **CRITICAL MEDICAL DISCLAIMER:**  
> This notebook, accompanying deep learning models, and associated web tools are engineered exclusively for academic research, educational demonstration, and preliminary screening support. They are **NOT** certified medical diagnostic devices. The predictions generated by these models must never replace clinical judgment, comprehensive dilated fundus examination, or professional diagnosis by a certified ophthalmologist or retinal specialist.

---

## 1. Project Overview & Clinical Background

**Diabetic Retinopathy (DR)** is a microvascular complication of diabetes mellitus and the leading cause of preventable blindness among working-age adults worldwide. Chronic hyperglycemia causes progressive damage to the delicate retinal microvasculature.

### International Clinical Diabetic Retinopathy (ICDR) Scale:
1. **Class 0 - No Diabetic Retinopathy**: Normal retinal vascular structure without detectable lesions.
2. **Class 1 - Mild NPDR**: Presence of microaneurysms only (small circular outpouchings in tiny capillaries).
3. **Class 2 - Moderate NPDR**: Presence of microaneurysms, dot-and-blot intraretinal hemorrhages, hard lipid exudates, and/or cotton wool spots.
4. **Class 3 - Severe NPDR**: High risk of rapid transition to blindness. Hallmark **4-2-1 rule**:
   - >20 intraretinal hemorrhages in each of the 4 retinal quadrants
   - Definite venous beading in 2 or more quadrants
   - Prominent intraretinal microvascular abnormalities (IRMA) in 1 or more quadrants
5. **Class 4 - Proliferative Diabetic Retinopathy (PDR)**: Advanced sight-threatening stage characterized by pathological neovascularization (abnormal fragile new vessel proliferation) and preretinal or vitreous hemorrhages.

### Objectives of this PBL Notebook:
- Implement standardized retinal image preprocessing (Ben Graham's illumination normalization & circular cropping).
- Handle severe medical class imbalance using balanced class weighting.
- Design a transfer learning pipeline utilizing MobileNetV2 and EfficientNetB0 backbones.
- Execute a two-phase training regimen (Feature Extraction followed by Fine-Tuning).
- Rigorously evaluate multi-class performance with Confusion Matrices, ROC-AUC, and F1-Scores.

In [ ]:
# Step 1: Environment Setup and Hardware Verification
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers, callbacks
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize

# Set seeds for scientific reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow Version : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Accelerated Hardware: GPU Detected -> {gpus[0].name}")
else:
    print("Accelerated Hardware: CPU Mode (If running in Colab, navigate to Runtime -> Change runtime type -> GPU)")

## 2. Directory Configuration and Dataset Verification

In [ ]:
# Determine project root
PROJECT_ROOT = Path(".").resolve().parent if Path("..").resolve().name == "diabetic-retinopathy-detection" else Path(".").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "No Diabetic Retinopathy",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative Diabetic Retinopathy"
]
CLASS_SHORT_NAMES = ["No DR (0)", "Mild (1)", "Moderate (2)", "Severe (3)", "Proliferative (4)"]
NUM_CLASSES = 5
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("Directory configuration verified.")

## 3. Retinal Fundus Preprocessing Pipeline

Standard RGB fundus photography presents two key optical challenges:
1. **Extraneous Black Camera Padding**: Wastes receptive field capacity in convolutional layers.
2. **Non-Uniform Spherical Illumination**: Spherical curvature of the eye produces center-to-periphery lighting gradients.

We implement:
- **Circular Retinal Cropping**: Eliminates non-informative margins.
- **Ben Graham's Retinal Illumination Normalization**: Subtraction of local Gaussian blur to accentuate fine microvascular lesions (microaneurysms and exudates).

In [ ]:
def crop_retina_circle(image_rgb: np.ndarray, tolerance: int = 10) -> np.ndarray:
    """Crops black outer borders around the spherical retina."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    mask = gray > tolerance
    if not np.any(mask):
        return image_rgb
    row_mask = np.any(mask, axis=1)
    col_mask = np.any(mask, axis=0)
    rmin, rmax = np.where(row_mask)[0][[0, -1]]
    cmin, cmax = np.where(col_mask)[0][[0, -1]]
    if rmax - rmin < 20 or cmax - cmin < 20:
        return image_rgb
    return image_rgb[rmin:rmax+1, cmin:cmax+1]

def apply_ben_graham_enhancement(image_rgb: np.ndarray, sigmaX: int = 10) -> np.ndarray:
    """Ben Graham's local Gaussian illumination correction."""
    blur = cv2.GaussianBlur(image_rgb, (0, 0), sigmaX)
    enhanced = cv2.addWeighted(image_rgb, 4, blur, -4, 128)
    return np.clip(enhanced, 0, 255).astype(np.uint8)

def preprocess_fundus_image(img_path: Path, target_size=(224, 224)) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (raw_resized, preprocessed_enhanced)."""
    pil_img = Image.open(str(img_path)).convert("RGB")
    raw_rgb = np.array(pil_img, dtype=np.uint8)
    raw_resized = cv2.resize(raw_rgb, target_size, interpolation=cv2.INTER_AREA)
    
    cropped = crop_retina_circle(raw_rgb)
    enhanced = apply_ben_graham_enhancement(cropped, sigmaX=10)
    enhanced_resized = cv2.resize(enhanced, target_size, interpolation=cv2.INTER_AREA)
    return raw_resized, enhanced_resized

print("Preprocessing functions defined successfully.")

## 4. Class Imbalance Mitigation & Balanced Weights

Retinal screening datasets often contain an order of magnitude more healthy cases than advanced stages. We compute inverse frequency class weights so minority stages (Class 3 & 4) contribute proportionally during backpropagation.

In [ ]:
def get_split_counts(split_dir: Path) -> dict:
    counts = {}
    for i in range(NUM_CLASSES):
        folder = split_dir / str(i)
        if folder.exists():
            counts[i] = len([p for p in folder.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
        else:
            counts[i] = 0
    return counts

train_counts = get_split_counts(TRAIN_DIR)
val_counts = get_split_counts(VAL_DIR)
test_counts = get_split_counts(TEST_DIR)

total_train = sum(train_counts.values())
print(f"Training set samples: {total_train}")

if total_train > 0:
    labels_expanded = []
    for cid, cnt in train_counts.items():
        labels_expanded.extend([cid] * cnt)
    
    unique_cls = np.unique(labels_expanded)
    raw_weights = compute_class_weight('balanced', classes=unique_cls, y=np.array(labels_expanded))
    class_weights = {int(c): float(w) for c, w in zip(unique_cls, raw_weights)}
    print("\nComputed Balanced Class Weights:")
    for cid, w in class_weights.items():
        print(f"  Class {cid} ({CLASS_SHORT_NAMES[cid]}): {w:.3f}")
else:
    class_weights = {i: 1.0 for i in range(NUM_CLASSES)}
    print("Notice: No training images found yet. Run utils/generate_samples.py to generate sample images.")

## 5. Medically-Realistic Data Augmentation

Data augmentation simulates physiological variations in patient positioning without introducing geometric distortions that would destroy tiny focal lesions.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=RANDOM_SEED),
    layers.RandomRotation(0.08, seed=RANDOM_SEED),       # Subtle camera rotation
    layers.RandomZoom(0.08, seed=RANDOM_SEED),           # Focal distance variation
    layers.RandomTranslation(0.05, 0.05, seed=RANDOM_SEED), # Retinal decentering
    layers.RandomContrast(0.10, seed=RANDOM_SEED)        # Pupil dilation differences
], name="retina_augmentation")

print("Data augmentation pipeline initialized.")

## 6. Model Architecture & Transfer Learning

We employ **MobileNetV2** (or **EfficientNetB0**) initialized with ImageNet pre-trained weights.

### Custom Classification Head:
- `GlobalAveragePooling2D()`: Flattens spatial features without parameter explosion.
- `BatchNormalization()`: Stabilizes activation distributions.
- `Dense(256, activation='relu', l2_reg=1e-4)`: Retinal biomarker representation.
- `Dropout(0.3)`: Mitigates overfitting.
- `Dense(5, activation='softmax')`: Outputs multi-class severity posterior probabilities.

In [ ]:
def create_retinacare_model(model_name="MobileNetV2", input_shape=(224, 224, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape, name="input_retina")
    
    if model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    elif model_name == "EfficientNetB0":
        base_model = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights="imagenet")
    else:
        raise ValueError(f"Unsupported model: {model_name}")
        
    base_model.trainable = False  # Freeze for initial Phase 1
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_feat")(x)
    x = layers.Dropout(0.3, name="dropout_1")(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="dense_sub")(x)
    x = layers.Dropout(0.15, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="severity_output")(x)
    
    model = models.Model(inputs, outputs, name=f"RetinaCare_{model_name}")
    return model, base_model

model, base_model = create_retinacare_model("MobileNetV2")
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Two-Stage Training Regimen

1. **Phase 1: Feature Extraction**: Backbone weights are frozen; only the custom classification head is trained.
2. **Phase 2: Fine-Tuning**: Top 30 layers of the backbone are unfrozen and trained with a reduced learning rate ($10^{-5}$).

In [ ]:
# Example training code (uncomment and execute when dataset is populated)
"""
# Callbacks setup
cb_checkpoint = callbacks.ModelCheckpoint(str(MODELS_DIR / 'best_model.keras'), monitor='val_loss', save_best_only=True, verbose=1)
cb_earlystop = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
cb_reducelr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

# Phase 1: Feature extraction
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=[cb_checkpoint, cb_earlystop, cb_reducelr]
)

# Phase 2: Fine tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    initial_epoch=len(history_p1.history['loss']),
    class_weight=class_weights,
    callbacks=[cb_checkpoint, cb_earlystop, cb_reducelr]
)
"""
print("Training configuration and callback templates prepared.")

## 8. Multi-Class Evaluation Metrics (Confusion Matrix & ROC-AUC)

Multi-class medical evaluation requires macro-averaged metrics to ensure diagnostic performance on severe cases is not masked by an abundance of healthy samples.

In [ ]:
def evaluate_predictions(y_true, y_pred_probs):
    """Generates confusion matrix and multi-class classification report."""
    y_pred = np.argmax(y_pred_probs, axis=1)
    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=CLASS_SHORT_NAMES, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(7, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_SHORT_NAMES, yticklabels=CLASS_SHORT_NAMES)
    plt.title("Diabetic Retinopathy Confusion Matrix")
    plt.xlabel("Predicted Severity")
    plt.ylabel("True Severity")
    plt.tight_layout()
    plt.show()

print("Evaluation functions ready.")

## 9. Conclusion & Web Deployment Integration

Once `models/best_model.keras` has been trained and saved, launch the interactive Streamlit clinical dashboard:

```bash
streamlit run app.py
```

The web application integrates:
- Automated Ben Graham preprocessing
- Interactive fundus image upload & side-by-side inspection
- Posterior probability visualization across all 5 clinical stages
- Automated clinical guidance & ophthalmologist follow-up recommendations
- Comprehensive model performance analytics